In [1]:
import sys, os
import subprocess as proc 
import textwrap
import inspect
import shutil
from datetime import datetime

In [5]:
import os

def list_directories(path):
    directories = []
    for entry in os.listdir(path):
        if os.path.isdir(os.path.join(path, entry)):
            directories.append(entry)
    return directories

# Specify the directory path you want to traverse
directory_path = "../PDB/"

# List all directories in the specified path
directories_list = list_directories(directory_path)

# Print the list of directories
print("List of directories:")
for directory in directories_list[:4]:
    print(directory)

List of directories:
1ehk
5z7e
1kbi
1ltd


In [7]:
#directories_list

In [8]:
pdb_list=["102m", "155c", "1a2f"]#, "1a2g", "1a6k", "1a6m", "1a6n", "1aa4", "1abs", "1ac4", "1ac8", "1aeb", "1aed", "1aee", "1aef"]
spin_states=["01", "05", "12", "16"]

In [23]:
class automation():
    def __init__(self):
        """ 
        Class that creates the necessary files for running jobs automatically except for the .xyz coordinate files. 
        """
        spin_states=["01", "05", "12", "16"]
        pdb_list=["1ccb", "1cce", "1ccj", "1a6m", '7cka']
        automation.__create_runtime_folders(pdb_list=pdb_list, spin_states=spin_states)
        for pdb_id in pdb_list:
            if not os.path.exists(f"../PDB/{pdb_id}/{pdb_id}_g16.xyz"):
                print(f"{pdb_id}_g16.xyz does not exist. Skipping {pdb_id}.")
                continue
            automation.__create_gaussian_scripts_all(pdb_id=pdb_id)
        return

    def __create_runtime_folders(pdb_list, spin_states):
        """
        Checks if folder names exist, if not, creates folders.  
        """
        for pdb_id in pdb_list:
            
            for spin_state in spin_states:
                if not os.path.exists(f"{pdb_id}"):
                    os.mkdir(f"{pdb_id}")
                if not os.path.exists(f"{pdb_id}/{pdb_id}{spin_state}/"):
                    os.mkdir(f"{pdb_id}/{pdb_id}{spin_state}/")
        return

    def __create_run_script_01(pdb_id):
        g16=f"""#!/bin/bash
        #SBATCH --time=12:00:00
        #SBATCH --partition=gpu-a100          
        #SBATCH --nodes=1
        #SBATCH -A beb00042                     
        #SBATCH --mem=32G                     
        #SBATCH --ntasks=72             
        #SBATCH --gres=gpu:4             

        module load cuda/11.8
        module load gaussian/16.C02
        module load anaconda3/2023.09

        g16 {pdb_id}01.com
        cd ../
        cp -r {pdb_id}01/ ../../../../scratch/projects/beb00042/{pdb_id}/
        python ../beb00042_agent.py "{pdb_id}" "01"

        """

        run01=inspect.cleandoc(g16)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}01/"):
            os.mkdir(f"{pdb_id}/{pdb_id}01/")
        run=open(f"{pdb_id}/{pdb_id}01/g16.sh", "w").write(run01)
        return
    
    def __create_run_script_05(pdb_id):
        g16=f"""#!/bin/bash
        #SBATCH --time=12:00:00
        #SBATCH --partition=gpu-a100          
        #SBATCH --nodes=1
        #SBATCH -A beb00042                     
        #SBATCH --mem32G                     
        #SBATCH --ntasks=72             
        #SBATCH --gres=gpu:4             

        module load cuda/11.8
        module load gaussian/16.C02

        g16 {pdb_id}05.com

        """

        run05=inspect.cleandoc(g16)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}05/"):
            os.mkdir(f"{pdb_id}/{pdb_id}05/")
        run=open(f"{pdb_id}/{pdb_id}05/g16.sh", "w").write(run05)
        return
    
    def __create_run_script_12(pdb_id):
        g16=f"""#!/bin/bash
        #SBATCH --time=12:00:00
        #SBATCH --partition=gpu-a100          
        #SBATCH --nodes=1
        #SBATCH -A beb00042                     
        #SBATCH --mem32G                     
        #SBATCH --ntasks=72             
        #SBATCH --gres=gpu:4             

        module load cuda/11.8
        module load gaussian/16.C02

        g16 {pdb_id}12.com

        """

        run12=inspect.cleandoc(g16)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}12/"):
            os.mkdir(f"{pdb_id}/{pdb_id}12/")
        run=open(f"{pdb_id}/{pdb_id}12/g16.sh", "w").write(run12)
        return
    
    def __create_run_script_16(pdb_id):
        g16=f"""#!/bin/bash
        #SBATCH --time=12:00:00
        #SBATCH --partition=gpu-a100          
        #SBATCH --nodes=1
        #SBATCH -A beb00042                     
        #SBATCH --mem32G                     
        #SBATCH --ntasks=72             
        #SBATCH --gres=gpu:4             

        module load cuda/11.8
        module load gaussian/16.C02

        g16 {pdb_id}16.com

        """

        run16=inspect.cleandoc(g16)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}16/"):
            os.mkdir(f"{pdb_id}/{pdb_id}16/")
        run=open(f"{pdb_id}/{pdb_id}16/g16.sh", "w").write(run16)
        return

    def __create_gaussian_scripts_01(pdb_id):
        link0=f"""%mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %chk={pdb_id}_II_S.chk
        # polar def2tzvp apfd int=(grid=ultrafine)

        {pdb_id}_II_S 

        0 1
        """

        link1=f"""

        --Link1--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %chk={pdb_id}_II_S.chk
        # apfd/def2TZVP int=(grid=ultrafine) geom=check guess=read pop=nbo scf=qc

        {pdb_id}_II_S_nbo

        0 1
        """

        link2=f""" 
        --Link2--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %chk={pdb_id}_II_S.chk
        # apfd/def2TZVP int=(grid=ultrafine) geom=check guess=read scrf=(smd,solvent=Chloroform) 

        {pdb_id}_II_S_solv 

        0 1 

        """

        

        with open(f"../PDB/{pdb_id}/{pdb_id}_g16.xyz", "r") as file:
            coords = file.read().splitlines()[2:]
        coords = '\n'.join(coords)
        com01=inspect.cleandoc(link0)+"\n"+coords+"\n\n"+inspect.cleandoc(link1)+"\n\n"+inspect.cleandoc(link2)+"\n\n\n"
        com=open(f"{pdb_id}/{pdb_id}01/{pdb_id}01.com", "w").write(com01)
        return

    def __create_gaussian_scripts_05(pdb_id):
        link3=f"""--Link3--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_II_S.chk
        %chk={pdb_id}_II_Q.chk
        # apfd/def2TZVP scf=maxcycle=999 geom=check pop=nbo

        {pdb_id}_II_Q

        0 5

        """

        link4=f"""--Link4--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %chk={pdb_id}_II_Q.chk
        # apfd/def2TZVP geom=check guess=read scrf=(smd,solvent=Chloroform)

        {pdb_id}_II_Q_solv

        0 5

        """

        com05=inspect.cleandoc(link3)+"\n\n"+inspect.cleandoc(link4)+"\n"
        com=open(f"{pdb_id}/{pdb_id}05/{pdb_id}05.com", "w").write(com05)
        return

    def __create_gaussian_scripts_12(pdb_id):
        link5=f"""--Link5--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_II_S.chk
        %chk={pdb_id}_III_D.chk
        # apfd/def2TZVP scf=maxcycle=999 geom=check pop=nbo

        {pdb_id}_III_D

        1 2

        """

        link6=f"""--Link6--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %chk={pdb_id}_III_D.chk
        # apfd/def2TZVP geom=check guess=read scrf=(smd,solvent=Chloroform)

        {pdb_id}_III_D_solv

        1 2

        """

        com12=inspect.cleandoc(link5)+"\n\n"+inspect.cleandoc(link6)+"\n"
        com=open(f"{pdb_id}/{pdb_id}12/{pdb_id}12.com", "w").write(com12)
        return
    
    def __create_gaussian_scripts_16(pdb_id):
        link7=f"""--Link7--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_III_D.chk
        %chk={pdb_id}_III_H.chk
        # apfd/def2TZVP scf=maxcycle=999 geom=check pop=nbo

        {pdb_id}_III_H

        1 6

        """

        link8=f"""--Link8--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %chk={pdb_id}_III_H.chk
        # apfd/def2TZVP geom=check guess=read scrf=(smd,solvent=Chloroform)

        {pdb_id}_III_H_solv

        1 6

        """

        com16=inspect.cleandoc(link7)+"\n\n"+inspect.cleandoc(link8)+"\n"
        com=open(f"{pdb_id}/{pdb_id}16/{pdb_id}16.com", "w").write(com16)
        return
    
    def __create_rerun_gaussian_scripts_05(pdb_id):
        link3re = f""" --Link3--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_II_S.chk
        %chk={pdb_id}_II_Sre.chk
        # apfd/gen scf=maxcycle=999 geom=check

        {pdb_id}_II_Q

        0 5

        C N O H 0
        6-31G*
        ****
        Fe 0
        TZVP
        ****


        """ 

        link3re1 = f""" --Link3--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_II_S.chk
        %chk={pdb_id}_II_Sre.chk
        # apfd/gen scf=maxcycle=999 geom=check

        {pdb_id}_II_Q

        0 5

        C N H 0
        6-31G*
        ****
        Fe 0
        TZVP
        ****


        """

        with open(f"../PDB/{pdb_id}/{pdb_id}_g16.xyz", "r") as file:
            coords = file.read().splitlines()[2:]
        coords_str = '\n'.join(coords)
        link_to_use = link3re if any("O" in line for line in coords) else link3re1
        com05re=inspect.cleandoc(link_to_use)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}05re/"):
            os.mkdir(f"{pdb_id}/{pdb_id}05re/")
        com=open(f"{pdb_id}/{pdb_id}05re/{pdb_id}05re.com", "w").write(com05re)
        return
    
    def __create_rerun_gaussian_scripts_12(pdb_id):
        link5re = f""" --Link5--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_II_S.chk
        %chk={pdb_id}_II_Sre.chk
        # apfd/gen scf=maxcycle=999 geom=check

        {pdb_id}_III_D

        1 2

        C N O H 0
        6-31G*
        ****
        Fe 0
        TZVP
        ****


        """ 

        link5re1 = f""" --Link5--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_II_S.chk
        %chk={pdb_id}_II_Sre.chk
        # apfd/gen scf=maxcycle=999 geom=check

        {pdb_id}_III_D

        1 2

        C N H 0
        6-31G*
        ****
        Fe 0
        TZVP
        ****


        """ 

        with open(f"../PDB/{pdb_id}/{pdb_id}_g16.xyz", "r") as file:
            coords = file.read().splitlines()[2:]
        coords_str = '\n'.join(coords)
        link_to_use = link5re if any("O" in line for line in coords) else link5re1
        com12re=inspect.cleandoc(link_to_use)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}12re/"):
            os.mkdir(f"{pdb_id}/{pdb_id}12re/")
        com=open(f"{pdb_id}/{pdb_id}12re/{pdb_id}12re.com", "w").write(com12re)
        return

    def __create_rerun_gaussian_scripts_16(pdb_id):
        link7re = f""" --Link7--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_III_D.chk
        %chk={pdb_id}_III_Dre.chk
        # apfd/gen scf=maxcycle=999 geom=check

        {pdb_id}_III_H

        1 6

        C N O H 0
        6-31G*
        ****
        Fe 0
        TZVP
        ****


        """ 

        link7re1 = f""" --Link7--
        %mem=128GB
        %CPU=0-1
        %gpucpu=0=0
        %oldchk={pdb_id}_III_D.chk
        %chk={pdb_id}_III_Dre.chk
        # apfd/gen scf=maxcycle=999 geom=check

        {pdb_id}_III_H

        1 6

        C N H 0
        6-31G*
        ****
        Fe 0
        TZVP
        ****


        """ 

        with open(f"../PDB/{pdb_id}/{pdb_id}_g16.xyz", "r") as file:
            coords = file.read().splitlines()[2:]
        coords_str = '\n'.join(coords)
        link_to_use = link7re if any("O" in line for line in coords) else link7re1
        com16re=inspect.cleandoc(link_to_use)+"\n"
        if not os.path.exists(f"{pdb_id}/{pdb_id}16re/"):
            os.mkdir(f"{pdb_id}/{pdb_id}16re/")
        com=open(f"{pdb_id}/{pdb_id}16re/{pdb_id}16re.com", "w").write(com16re)
        return

    def __create_gaussian_scripts_all(pdb_id):
        automation.__create_gaussian_scripts_01(pdb_id=pdb_id)
        automation.__create_gaussian_scripts_05(pdb_id=pdb_id)
        automation.__create_gaussian_scripts_12(pdb_id=pdb_id)
        automation.__create_gaussian_scripts_16(pdb_id=pdb_id)
        automation.__create_rerun_gaussian_scripts_05(pdb_id=pdb_id)
        automation.__create_rerun_gaussian_scripts_12(pdb_id=pdb_id)
        automation.__create_rerun_gaussian_scripts_16(pdb_id=pdb_id)
        automation.__create_run_script_01(pdb_id=pdb_id)
        automation.__create_run_script_05(pdb_id=pdb_id)
        automation.__create_run_script_12(pdb_id=pdb_id)
        automation.__create_run_script_16(pdb_id=pdb_id)
        return
        



In [21]:
inst = automation()

In [5]:
if not os.path.isfile("history.txt"):
    open("history.txt", "w").write(f"HISTORY OF CALCULATIONS FOR REDOX HEME PROJECT\nMROGISNKI, GENSCH, JONES, ZHARTOVSKA, LEICH, GUO\n\n")

In [6]:
def __error_capturing(logfile, basedir, workdir, spin_state, pdb_id):
    if " Convergence failure -- run terminated." in logfile[-10:0]: 
        os.chdir(basedir)
        logmsg = f""" 
        Time, Date
        {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
        Path
        {workdir}
        convergence failure in {spin_state} - inspection necessary. 
        -----------------------------------------------------------------------------------------------------------------
        \n
        """
        open("history.txt", "a").write(inspect.cleandoc(logmsg)+"\n\n")
        if spin_state == "05" or spin_state == "12":
            shutil.copyfile(f"{pdb_id}/{pdb_id}01/{pdb_id}_II_S.chk", f"{pdb_id}/{pdb_id}{spin_state}re/{pdb_id}_II_S.chk")
        if spin_state == "16":
            shutil.copyfile(f"{pdb_id}/{pdb_id}12/{pdb_id}_III_D.chk", f"{pdb_id}/{pdb_id}{spin_state}re/{pdb_id}_III_D.chk")
        os.chdir(f"{pdb_id}/{pdb_id}{spin_state}re")
        runre = proc.run(["qg16", f"{pdb_id}{spin_state}re"])
    
    elif " Route card not found." in logfile[-10:0]:
        os.chdir(basedir)
        errormsg = f""" 
        Time, Date
        {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
        Path
        {workdir}
        no input in {spin_state} - inspection necessary. 
        -----------------------------------------------------------------------------------------------------------------
        \n
        """
        open("history.txt", "a").write(inspect.cleandoc(errormsg)+"\n\n")
    
    elif " Input Error" in logfile[-10:0]:
        os.chdir(basedir)
        errormsg = f""" 
        Time, Date
        {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
        Path
        {workdir}
        cannot read input in {spin_state} - inspection necessary. 
        -----------------------------------------------------------------------------------------------------------------
        """
        open("history.txt", "a").write(inspect.cleandoc(errormsg)+"\n\n")
    
    elif "Symbol not recognized in MSubst" in logfile[-10:0]:
        os.chdir(basedir)
        errormsg = f""" 
        Time, Date
        {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
        Path
        {workdir}
        unrecognized atomic symbol in {spin_state} - inspection necessary. 
        -----------------------------------------------------------------------------------------------------------------
        """
        open("history.txt", "a").write(inspect.cleandoc(errormsg)+"\n\n")
    
    elif " Error termination request processed by link 9999." in logfile[-10:0]:
        os.chdir(basedir)
        errormsg = f""" 
        Time, Date
        {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
        Path
        {workdir}
        geometry not converged in {spin_state} - inspection necessary. 
        -----------------------------------------------------------------------------------------------------------------
        """
        open("history.txt", "a").write(inspect.cleandoc(errormsg)+"\n\n")
    
    elif " Old file " and " could not be opened." in logfile[-10:0]:
        os.chdir(basedir)
        errormsg = f""" 
        Time, Date
        {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
        Path
        {workdir}
        chk can not be read in {spin_state} - inspection necessary. 
        -----------------------------------------------------------------------------------------------------------------
        """
        open("history.txt", "a").write(inspect.cleandoc(errormsg)+"\n\n")
    #elif aborted due to time restraint: figure out which jobs were not done in time based on results not put into results csv.
    else: pass
    return

In [7]:
pdb_id="2gsm"
basedir = "/home/pbuser/Desktop/PhD_WORK/automation_heme/"
workdirs=[f"{pdb_id}/{pdb_id}01", f"{pdb_id}/{pdb_id}05", f"{pdb_id}/{pdb_id}12", f"{pdb_id}/{pdb_id}16"]
run01 = proc.run(["qg16", f"{pdb_id}01"], capture_output=True, text=True)
#wait 2000000
for workdir in workdirs:
    os.chdir(workdir)
    
    if workdir[-2:0] == "01":
        logfile=open(workdir+"Erfolgreich.log", "r").readlines()
        if "Normal termination of Gaussian 16" in logfile[-1]:
            shutil.copyfile(f"{pdb_id}/{pdb_id}01/{pdb_id}_II_S.chk", f"{pdb_id}/{pdb_id}05/{pdb_id}_II_S.chk")
            shutil.copyfile(f"{pdb_id}/{pdb_id}01/{pdb_id}_II_S.chk", f"{pdb_id}/{pdb_id}12/{pdb_id}_II_S.chk")
            os.chdir(f"../{pdb_id}05")
            run05 = proc.run(["qg16", f"{pdb_id}05"], capture_output=True, text=True)
            os.chdir(f"../{pdb_id}12")
            run12 = proc.run(["qg16", f"{pdb_id}12"], capture_output=True, text=True)
        __error_capturing(logfile=logfile, basedir=basedir, workdir=workdir, spin_state="01")
    
    elif workdir[-2:0] == "05":
        logfile=open(workdir+"Erfolgreich.log", "r").readlines()
        if "Normal termination of Gaussian 16" in logfile[-1]:
            os.chdir(basedir)
            logmsg = f""" 
            Time, Date
            {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
            Path
            {workdir}
            {spin_state} - Done! 
            -----------------------------------------------------------------------------------------------------------------
            """
            open("history.txt", "a").write(inspect.cleandoc(logmsg)+"\n\n")
        __error_capturing(logfile=logfile, basedir=basedir, workdir=workdir, spin_state="05")
    
    elif workdir[-2:0] == "12":
        logfile=open(workdir+"Erfolgreich.log", "r").readlines()
        if "Normal termination of Gaussian 16" in logfile[-1]:
            shutil.copyfile(f"{pdb_id}/{pdb_id}12/{pdb_id}_III_D.chk", f"{pdb_id}/{pdb_id}16/{pdb_id}_III_D.chk")
            os.chdir(f"../{pdb_id}16")
            run12 = proc.run(["qg16", f"{pdb_id}16"], capture_output=True, text=True)
            os.chdir(basedir)
            logmsg = f""" 
            Time, Date
            {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
            Path
            {workdir}
            {spin_state} - Done! 
            -----------------------------------------------------------------------------------------------------------------
            """
            open("history.txt", "a").write(inspect.cleandoc(logmsg)+"\n\n")
        __error_capturing(logfile=logfile, basedir=basedir, workdir=workdir, spin_state="12")
    
    elif workdir[-2:0] == "16":
        logfile=open(workdir+"Erfolgreich.log", "r").readlines()
        if "Normal termination of Gaussian 16" in logfile[-1]:
            shutil.copyfile(f"{pdb_id}/{pdb_id}12/{pdb_id}_III_D.chk", f"{pdb_id}/{pdb_id}16/{pdb_id}_III_D.chk")
            os.chdir(f"../{pdb_id}16")
            os.chdir(basedir)
            logmsg = f""" 
            Time, Date
            {datetime.now().strftime("%H:%M:%S")}, {datetime.date.today()}
            Path
            {workdir}
            {pdb_id}{spin_state} - Done! 
            -----------------------------------------------------------------------------------------------------------------
            """
            open("history.txt", "a").write(inspect.cleandoc(logmsg)+"\n\n")
        __error_capturing(logfile=logfile, basedir=basedir, workdir=workdir, spin_state="16")


FileNotFoundError: [Errno 2] No such file or directory: 'qg16'